In [ ]:
# %matplotlib widget
import os, sys, json
from tqdm import tqdm
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.colors import TwoSlopeNorm
from matplotlib.ticker import FixedFormatter
import seaborn as sns
import numpy as np
import pandas as pd

sys.path.append(os.path.abspath(".."))
import mienc.support as ms
from mienc import Corrector, NonLinearEstimator

from scipy.io import loadmat, savemat
from scipy.stats import ks_1samp, uniform, binom, pearsonr, spearmanr
from scipy.spatial import Voronoi, voronoi_plot_2d

from nilearn import datasets, plotting, image
import nibabel as nib

sns.set_theme("poster", "ticks")
DATA_DIR = "/mnt/DATA/Giulio_NonLinearityResults/ReRun/fMRI_region_size/"

In [ ]:
aal_atlas_centers = pd.read_excel(os.path.join(DATA_DIR, "AAL_regions.xls"))
aal_atlas_centers_labels = (
    aal_atlas_centers["<b>Labels</b>"].apply(lambda x: x.split()[1]).tolist()
)
atlas_data = datasets.fetch_atlas_aal()
atlas_filename = atlas_data.maps
region_indices = {l: i for l, i in zip(atlas_data["labels"], atlas_data["indices"])}

reg_loc = []
image_data = image.get_data(atlas_filename)
affine = image.load_img(atlas_filename).affine
for l in region_indices:
    if l in aal_atlas_centers_labels:
        reg_loc.append(np.where(image_data == int(region_indices[l])))

div_palette = sns.color_palette("vlag", as_cmap=True)# plt.cm.seismic
div_palette.set_over(div_palette.get_bad())
div_palette.set_under(color=[0.5,0.5,0.5,0.5])
div_palette.set_bad(color="grey")

def plot_brain(
    in_array,
    title=None,
    cut_coords=None,
    axes=None,
    cmap=div_palette,
    vmin=None,
    vmax=None,
    norm=None,
):
    MAXINT = np.iinfo(np.int32).max

    if norm is None:
        if vmin is None:
            vmin = np.nanmin(in_array)
        if vmax is None:
            vmax = np.nanmax(in_array)
        normalised = (in_array-vmin)/(vmax-vmin)
    else:
        normalised=norm(in_array)
    normalised[np.isinf(normalised)]=1
    values = np.full_like(image_data, np.nan, dtype=np.int32)
    for i in range(90):
        values[reg_loc[i][0], reg_loc[i][1], reg_loc[i][2]] = normalised[i] * MAXINT

    img = nib.nifti1.Nifti1Image(values, affine)
    plotting.plot_roi(
        img,
        title=title,
        cut_coords=cut_coords,
        cmap=cmap,
        axes=axes,
        vmin=0,
        vmax=MAXINT
    )


def plot_cap(
    in_array,
    title=None,
    region_positions=None,
    axes=None,
    cmap=div_palette,
    vmin=None,
    vmax=None,
    plane_distance=10,
):
    source_z = region_positions.z.min()
    diameter = region_positions.z.max() - source_z
    region_positions["XP"] = (
        region_positions.x
        / (region_positions.z - source_z + plane_distance)
        * (diameter + plane_distance)
    )
    region_positions["YP"] = (
        region_positions.y
        / (region_positions.z - source_z + plane_distance)
        * (diameter + plane_distance)
    )
    if vmin is None:
        vmin = np.nanmin(in_array)
    if vmax is None:
        vmax = np.nanmax(in_array)
    vor = Voronoi(
        np.concatenate(
            [
                region_positions[["XP", "YP"]],
                np.transpose(
                    [
                        22 * np.sin(np.linspace(0, 2 * np.pi, 100)),
                        22 * np.cos(np.linspace(0, 2 * np.pi, 100)) - 0.5,
                    ]
                ),
            ]
        )
    )
    normalised = (in_array - vmin) / (vmax - vmin)
    if axes is not None:
        plt.sca(axes)
    plt.gca().set_aspect("equal")

    for i, v in enumerate(normalised):
        region = vor.regions[vor.point_region[i]]
        vertices = np.array([vor.vertices[n] for n in region])
        plt.fill(vertices[:, 0], vertices[:, 1], color=cmap(v))
    if title is not None:
        plt.title(title)

def HolmThresholdFromP(p_values: np.ndarray):
    """Returns the Holm-Bonferroni threshold given an array of p-values.
    NOTA BENE: reject the null hypotheses when **strictly smaller** than this thresold. This corresponds to the *p-value* of the first non-rejected null hypothesis.

    Parameters
    ----------
    p_values : np.ndarray
        The *p-values* to consider, can be N-dimensional, will be flattened.

    Returns
    -------
    float
        The *p-value* of the first non-rejected hypothesis.
    """
    sorted_p = np.sort(p_values.flatten())
    good = sorted_p < 0.05 / (sorted_p.size - np.arange(sorted_p.size))
    which = np.argmin(good)

    if which == 0 and sorted_p[-1] < 0.05:
        return 0.05

    return sorted_p[which]

In [ ]:
def compute_localised_non_linearity(
    results_file,
    subset_identifiers,
    output_prefix,
    pair_num,
    subj_num,
    nbins,
    nsteps,
):
    corrct = Corrector(
        nbins,
        nsteps,
        folder_name=os.path.dirname(results_file).format(subset_identifiers[0]),
        config="../config.ini",
        cache_dir="../cache/",
    )
    corrct.compute_correction()
    for subset_id in subset_identifiers:

        if not (os.path.isfile(
            f"/mnt/DATA/Giulio_NonLinearityResults/ReRun/Results/localised/{output_prefix}_ks_stat_{subset_id}.npy"
        ) and os.path.isfile(
            f"/mnt/DATA/Giulio_NonLinearityResults/ReRun/Results/localised/{output_prefix}_ks_p_{subset_id}.npy"
        )):
            if not os.path.isfile(
                f"/mnt/DATA/Giulio_NonLinearityResults/ReRun/Results/localised/{output_prefix}_region_quantiles_{subset_id}.npy"
            ):
                region_quantiles = np.zeros((subj_num, pair_num))
                for s in tqdm(range(subj_num), "Subject"):
                    pat = np.load(results_file.format(subset_id, s))
                    true = pat[:, 0]
                    surr = np.sort(pat[:, 1:], 1)
                    for i in range(pair_num):
                        region_quantiles[s, i] = np.searchsorted(
                            surr[i, :], true[i], "left"
                        )
                    np.save(
                        f"/mnt/DATA/Giulio_NonLinearityResults/ReRun/Results/localised/{output_prefix}_region_quantiles_{subset_id}.npy",
                        region_quantiles,
                    )
            ks_stat = np.zeros(pair_num)
            ks_p = np.zeros(pair_num)
            for i in tqdm(range(pair_num), "Pair"):
                stat, pval = ks_1samp(
                    region_quantiles[:, i], uniform.cdf, (0, 100), alternative="less"
                )
                ks_stat[i] = stat
                ks_p[i] = pval
            np.save(
                f"/mnt/DATA/Giulio_NonLinearityResults/ReRun/Results/localised/{output_prefix}_ks_stat_{subset_id}.npy",
                ks_stat,
            )
            np.save(
                f"/mnt/DATA/Giulio_NonLinearityResults/ReRun/Results/localised/{output_prefix}_ks_p_{subset_id}.npy",
                ks_p,
            )


      

In [ ]:
def show_localised_non_linearity(
    results_file,
    subset_description,
    subset_identifiers,
    subset_names,
    output_prefix,
    pair_num,
    elec_num,
    region_positions,
    cut_position,
):
    global MAX_CM, MAX_CM2
    for subset_id, subset_na in zip(subset_identifiers, subset_names):
        ks_stat = np.load(
            f"/mnt/DATA/Giulio_NonLinearityResults/ReRun/Results/localised/{output_prefix}_ks_stat_{subset_id}.npy"
        )
        ks_p = np.load(
            f"/mnt/DATA/Giulio_NonLinearityResults/ReRun/Results/localised/{output_prefix}_ks_p_{subset_id}.npy"
        )

        ks_stat_sha = np.load(
            f"/mnt/DATA/Giulio_NonLinearityResults/ReRun/Results/localised/{output_prefix+'_sha'}_ks_stat_{subset_id}.npy"
        )
        ks_p_sha = np.load(
            f"/mnt/DATA/Giulio_NonLinearityResults/ReRun/Results/localised/{output_prefix+'_sha'}_ks_p_{subset_id}.npy"
        )
        thresh = HolmThresholdFromP(np.concatenate([ks_p, ks_p_sha]))
        print(thresh)
        corrected = np.full(pair_num, np.nan)
        corrected[ks_p < thresh] = ks_stat[ks_p < thresh]
        sig_pair = np.zeros([elec_num, elec_num])
        sig_pair[np.triu_indices(elec_num, 1)] = corrected
        sig_pair += sig_pair.T
        np.fill_diagonal(sig_pair, np.nan)
        siginreg = np.sum(sig_pair > 0, 1)
        print("Non linear connections:", np.sum(siginreg) / 2)

        # thresh_sha = HolmThresholdFromP(ks_p_sha)

        corrected_sha = np.full(pair_num, np.nan)
        corrected_sha[ks_p_sha < thresh] = ks_stat_sha[ks_p_sha < thresh]
        sig_pair_sha = np.zeros([elec_num, elec_num])
        sig_pair_sha[np.triu_indices(elec_num, 1)] = corrected_sha
        sig_pair_sha += sig_pair_sha.T
        np.fill_diagonal(sig_pair_sha, np.nan)
        siginreg_sha = np.sum(sig_pair_sha > 0, 1)
        print("Non linear connections:", np.sum(siginreg_sha) / 2)

        fix, ax = plt.subplots(
            1, 3, gridspec_kw={"width_ratios": [4, 4, 0.2]}, figsize=(12, 7)
        )
        vmax = max(np.nanmax(sig_pair), np.nanmax(sig_pair_sha))
        vmin = min(np.nanmin(sig_pair), np.nanmin(sig_pair_sha))
        plt.sca(ax[0])
        plt.imshow(sig_pair, interpolation="none", vmax=vmax, vmin=vmin)
        plt.yticks(
            np.arange(0, elec_num, 10), np.arange(elec_num)[::10]
        )
        step = 10 if len(np.arange(0, sum(siginreg > 0), 10)) < 12 else 20
        plt.xticks(
            np.arange(0, elec_num, step),
            np.arange(elec_num)[::step],
        )
        plt.xlabel(f"Region")
        plt.title("True")

        plt.sca(ax[1])
        plt.imshow(sig_pair_sha, interpolation="none", vmax=vmax, vmin=vmin)
        plt.yticks(
            np.arange(0, elec_num, 10), np.arange(elec_num)[::10]
        )
        step = 10 if len(np.arange(0, elec_num, 10)) < 12 else 20
        plt.xticks(
            np.arange(0, elec_num, step),
            np.arange(elec_num)[::step],
        )
        plt.xlabel(f"Region")
        plt.title("Shadow")
        plt.colorbar(ax=ax[1], cax=ax[2], shrink=0.2).ax.set_ylabel("KS statistcs", rotation=90)

        plt.suptitle(
            subset_description
            + subset_na
            + f" - {np.sum(ks_p<thresh)} ({100*np.sum(ks_p<thresh)/pair_num:.3} %) significant pairs"
        )
        plt.show()

        if region_positions is not None and (siginreg > 0).any():
            
            print("Siginreg max:", np.max(siginreg))
            if "aal" in results_file:
                fig, ax = plt.subplots(
                    3,1, gridspec_kw={"height_ratios": [4,4, 0.5]}, figsize=(8, 8)
                )
                sc = ax[2].scatter([np.nan],[np.nan],c=0,cmap=div_palette,norm=TwoSlopeNorm(vmin=0, vcenter=elec_num/2, vmax=elec_num-1))
                cbar = plt.colorbar(sc, cax=ax[2], shrink=0.35,ticks=[0,elec_num/2, elec_num-1],location="bottom",
                        format=FixedFormatter(["no\nnon-linear\nconnections","significantly\nabove random\ngraph","complete\nnon-linear\nconnections"]))
                        # format=FixedFormatter(["no\nnon-linear\nconnections","random\ngraph\naverage","complete\nnon-linear\nconnections"]))
                
                dist = binom(elec_num-1, np.sum(siginreg)/(elec_num*(elec_num-1)))
                sig_rg = dist.ppf(1-0.05/(2*elec_num))
                norm = TwoSlopeNorm(vmin=-0.0001, vcenter= sig_rg if np.sum(siginreg)>0 else elec_num/2, vmax=elec_num-1)
                print(region_positions.loc[siginreg>=sig_rg, ["Labels","y"]])
                print(siginreg[siginreg>=sig_rg])
                print(np.sum(siginreg)/elec_num, sig_rg ,elec_num-1)
                # norm = TwoSlopeNorm(vmin=-0.0001, vcenter=np.sum(siginreg)/elec_num if np.sum(siginreg)>0 else elec_num/2, vmax=elec_num-1)
                plot_brain(
                    siginreg,
                    "AAL 90 regions - True",
                    cut_position,
                    ax[0],
                    div_palette,
                    norm = norm,
                )  # (-15,-75,27)

                dist = binom(elec_num-1, np.sum(siginreg_sha)/(elec_num*(elec_num-1)))
                sig_rg = dist.ppf(1-0.05/(2*elec_num))
                norm = TwoSlopeNorm(vmin=-0.0001, vcenter= sig_rg if np.sum(siginreg_sha)>0 else elec_num/2, vmax=elec_num-1)
                print(region_positions.loc[siginreg_sha>=sig_rg, ["Labels","y"]])
                print(siginreg_sha[siginreg_sha>=sig_rg])
                print(np.sum(siginreg_sha)/elec_num, sig_rg ,elec_num-1)
                plot_brain(
                    siginreg_sha,
                    "AAL 90 regions - Shadow",
                    cut_position,
                    ax[1],
                    div_palette,
                    norm = norm,
                ) 

            else:
                fig, ax = plt.subplots(
                    1, 3, gridspec_kw={"width_ratios": [4,4, 0.4]}, figsize=(8, 6)
                )
                vmax = max(np.max(siginreg), 10)
                sc = plt.scatter(
                    [np.nan], [np.nan], c=0, cmap=div_palette, vmin=1, vmax=vmax
                )
                cbar = plt.colorbar(sc, cax=ax[2], shrink=0.4, extend="min")
                cbar.set_label("# significantly non linear relationships")
                plot_cap(
                    siginreg,
                    output_prefix
                    + " - "
                    + subset_description
                    + subset_na
                    + " - Non-Linearity position",
                    region_positions,
                    ax[0],
                    div_palette,
                    1,
                    vmax,
                )
            plt.savefig(f"{output_prefix}_brain.pdf", bbox_inches="tight")
            plt.show()


In [ ]:
pair_num = 4005
subj_num = 245
elec_num = 90
quantiles = [0.6, 0.9]
cut_pos = (-15, -75, 27)
results_file = "/mnt/DATA/Giulio_NonLinearityResults/ReRun/Results/eso245_aal_{}_90_bin7/subject{:02}_7.npy"
subset_description = "fMRI AAL90 "
subset_identifiers = ["raw", "mod", "strin"]  #["raw"]# 
subset_names = ["raw", "mod", "strin"]  # ["strin"]#
output_prefix = "fMRI_AAL90"
region_positions = pd.read_excel(os.path.join(DATA_DIR, "AAL_regions.xls")).loc[
    slice(None), ["<b>Labels</b>", "<b>X</b>", "<b>Y</b>", "<b>Z</b>"]
]
region_positions.rename(columns={"<b>Labels</b>":"Labels", "<b>X</b>":"x", "<b>Y</b>":"y", "<b>Z</b>":"z"}, inplace=True)
region_positions["Labels"] = region_positions["Labels"].apply(lambda x: x.split()[1])
nbins = 7
with open(os.path.join(os.path.dirname(results_file.format("strin",0)), "shape.json")) as fp:
    nsteps = json.load(fp)[0]
compute_localised_non_linearity(
    results_file,
    subset_identifiers,
    output_prefix,
    pair_num,
    subj_num,
    nbins,
    nsteps,
)
show_localised_non_linearity(
    results_file,
    subset_description,
    subset_identifiers,
    subset_names,
    output_prefix,
    pair_num,
    elec_num,
    region_positions,
    cut_pos,
)

In [ ]:
import zipfile
zipfile.PyZipFile().extract

In [ ]:
from colorspacious import cspace_converter
converters = {}

_deuter50_space = {"name": "sRGB1+CVD",
                    "cvd_type": "deuteranomaly",
                    "severity": 50}
converters["deuter50"] = cspace_converter(_deuter50_space, "sRGB1")
_deuter100_space = {"name": "sRGB1+CVD",
                    "cvd_type": "deuteranomaly",
                    "severity": 100}
converters["deuter100"] = cspace_converter(_deuter100_space, "sRGB1")

_prot50_space = {"name": "sRGB1+CVD",
                    "cvd_type": "protanomaly",
                    "severity": 50}
converters["prot50"] = cspace_converter(_prot50_space, "sRGB1")

_prot100_space = {"name": "sRGB1+CVD",
                    "cvd_type": "protanomaly",
                    "severity": 100}
converters["prot100"] = cspace_converter(_prot100_space, "sRGB1")

In [ ]:
import colorblindnessCool as cc
fig = plt.figure(figsize=(15, 10))
gs = fig.add_gridspec(3,3,width_ratios=[5, 5, 0.2])
ax = np.zeros((3,2),dtype=object)
for i in range(3):
    for j in range(2):
        ax[i,j] = fig.add_subplot(gs[i, j])
ax_c = fig.add_subplot(gs[:, 2])
# fig, ax = plt.subplots(4, 2, gridspec_kw={"height_ratios": [4, 4, 4, 0.5]}, figsize=(8, 8))
plt.subplots_adjust(wspace=0.05, hspace=-0.)
sc = ax_c.scatter(
    [np.nan],
    [np.nan],
    c=0,
    cmap=div_palette,
    norm=TwoSlopeNorm(vmin=0, vcenter=(elec_num-1) / 2, vmax=elec_num - 1),
)
cbar = plt.colorbar(
    sc,
    cax=ax_c,
    shrink=0.20,
    ticks=[0, elec_num / 2, elec_num - 1],
    location="right",
    format=FixedFormatter(
        [
            "no\nnon-linear\nconnections",
            "significantly\nabove\nrandom graph",
            "complete\nnon-linear\nconnections",
        ]
    )
)
for ha, tick in zip(["left", "center", "right"], ax_c.yaxis.get_majorticklabels()):
    tick.set_horizontalalignment(ha)
    tick.set_verticalalignment("top")
    tick.set_rotation(90)
    tick.set_rotation_mode("anchor")

full_names={"raw":"Raw", "mod":"Moderate", "strin":"Stringent"}
for sn, subset_id in enumerate(["raw", "mod", "strin"]):
    ks_stat = np.load(
        f"/mnt/DATA/Giulio_NonLinearityResults/ReRun/Results/localised/{output_prefix}_ks_stat_{subset_id}.npy"
    )
    ks_p = np.load(
        f"/mnt/DATA/Giulio_NonLinearityResults/ReRun/Results/localised/{output_prefix}_ks_p_{subset_id}.npy"
    )

    ks_stat_sha = np.load(
        f"/mnt/DATA/Giulio_NonLinearityResults/ReRun/Results/localised/{output_prefix+'_sha'}_ks_stat_{subset_id}.npy"
    )
    ks_p_sha = np.load(
        f"/mnt/DATA/Giulio_NonLinearityResults/ReRun/Results/localised/{output_prefix+'_sha'}_ks_p_{subset_id}.npy"
    )
    thresh = HolmThresholdFromP(np.concatenate([ks_p, ks_p_sha]))
    print(thresh)
    corrected = np.full(pair_num, np.nan)
    corrected[ks_p < thresh] = ks_stat[ks_p < thresh]
    sig_pair = np.zeros([elec_num, elec_num])
    sig_pair[np.triu_indices(elec_num, 1)] = corrected
    sig_pair += sig_pair.T
    np.fill_diagonal(sig_pair, np.nan)
    siginreg = np.sum(sig_pair > 0, 1)
    print("Non linear connections:", np.sum(siginreg) / 2)

    # thresh_sha = HolmThresholdFromP(ks_p_sha)

    corrected_sha = np.full(pair_num, np.nan)
    corrected_sha[ks_p_sha < thresh] = ks_stat_sha[ks_p_sha < thresh]
    sig_pair_sha = np.zeros([elec_num, elec_num])
    sig_pair_sha[np.triu_indices(elec_num, 1)] = corrected_sha
    sig_pair_sha += sig_pair_sha.T
    np.fill_diagonal(sig_pair_sha, np.nan)
    siginreg_sha = np.sum(sig_pair_sha > 0, 1)

    dist = binom(elec_num - 1, np.sum(siginreg) / (elec_num * (elec_num - 1)))
    sig_rg = dist.ppf(1 - 0.05 / (2 * elec_num))
    norm = TwoSlopeNorm(
        vmin=-0.0001,
        vcenter=sig_rg if np.sum(siginreg) > 0 else elec_num / 2,
        vmax=elec_num - 1,
    )
    # print(region_positions.loc[siginreg >= sig_rg, ["Labels", "y"]])
    # print(siginreg[siginreg >= sig_rg])
    # print(np.sum(siginreg) / elec_num, sig_rg, elec_num - 1)
    # norm = TwoSlopeNorm(vmin=-0.0001, vcenter=np.sum(siginreg)/elec_num if np.sum(siginreg)>0 else elec_num/2, vmax=elec_num-1)
    plot_brain(
        siginreg,
        full_names[subset_id],
        cut_pos,
        ax[sn,0],
        div_palette,
        norm=norm,
    )  # (-15,-75,27)

    dist = binom(elec_num - 1, np.sum(siginreg_sha) / (elec_num * (elec_num - 1)))
    sig_rg = dist.ppf(1 - 0.05 / (2 * elec_num))
    norm = TwoSlopeNorm(
        vmin=-0.0001,
        vcenter=sig_rg if np.sum(siginreg_sha) > 0 else elec_num / 2,
        vmax=elec_num - 1,
    )
    print(subset_id, pearsonr(siginreg, siginreg_sha))
    # print(region_positions.loc[siginreg_sha >= sig_rg, ["Labels", "y"]])
    # print(siginreg_sha[siginreg_sha >= sig_rg])
    # print(np.sum(siginreg_sha) / elec_num, sig_rg, elec_num - 1)
    plot_brain(
        siginreg_sha,
        None,
        cut_pos,
        ax[sn,1],
        div_palette,
        norm=norm,
    )
    converter=converters["prot100"]
    for patch in ax[sn,0].patches:
        fc = patch.get_facecolor()[:3]
        patch.set_facecolor(np.clip(converter(fc), 0, 1))
    for patch in ax[sn,1].patches:
        fc = patch.get_facecolor()[:3]
        patch.set_facecolor(np.clip(converter(fc), 0, 1))
    ax[sn,0].set_ylabel(full_names[subset_id])
ax[0,0].set_title("True", pad=-50)
ax[0,1].set_title("Shadow", pad=-50)
# plt.savefig(f"localisation_fMRI2.pdf", bbox_inches="tight")
cc.get_image()
plt.show()
cc.show_transformed()

In [ ]:
from matplotlib.backends.backend_agg import FigureCanvasAgg as FigureCanvas
from matplotlib.figure import Figure

subset_id="strin"
ks_stat = np.load(
    f"/mnt/DATA/Giulio_NonLinearityResults/ReRun/Results/localised/{output_prefix}_ks_stat_{subset_id}.npy"
)
ks_p = np.load(
    f"/mnt/DATA/Giulio_NonLinearityResults/ReRun/Results/localised/{output_prefix}_ks_p_{subset_id}.npy"
)

thresh = HolmThresholdFromP(ks_p)

corrected = np.full(pair_num, np.nan)
corrected[ks_p < thresh] = ks_stat[ks_p < thresh]
sig_pair = np.zeros([elec_num, elec_num])
sig_pair[np.triu_indices(elec_num, 1)] = corrected
sig_pair += sig_pair.T
np.fill_diagonal(sig_pair, np.nan)
siginreg = np.sum(sig_pair > 0, 1)
dist = binom(elec_num-1, np.sum(siginreg)/(elec_num*(elec_num-1)))
vamx = dist.ppf(1-0.05/90)
norm = TwoSlopeNorm(vmin=-0.001, vcenter=np.sum(siginreg)/elec_num if np.sum(siginreg)>0 else elec_num/2, vmax=vamx)

div_palette = plt.cm.seismic
div_palette.set_under(div_palette.get_bad())
div_palette.set_over(color="green")
div_palette.set_bad(color="grey")
div_palette.set_under(color="grey")

MAXINT = np.iinfo(np.int32).max

normalised=norm(siginreg)
print(normalised)
FACTOR = MAXINT
values = np.full_like(image_data, np.nan, dtype=np.int32)
for i in range(90):
    values[reg_loc[i][0], reg_loc[i][1], reg_loc[i][2]] = normalised[i] * FACTOR

img = nib.nifti1.Nifti1Image(values, affine)


fig = Figure()
canvas = FigureCanvas(fig)
ax = fig.gca()
plotting.plot_roi(
    img,
    title="bnla",
    cut_coords=cut_pos,
    cmap=div_palette,
    vmin=-1,
    vmax=FACTOR,
    axes=ax
)

# ax.text(0.0,0.0,"Test", fontsize=45)
# ax.axis('off')

canvas.draw()       # draw the canvas, cache the renderer

image = np.array(canvas.buffer_rgba(), dtype='uint8')

print(np.nanmean(values), np.nanmax(values), MAXINT, np.nanmin(values[values>-2147483648]), normalised[normalised*FACTOR<0])
plt.show()


In [ ]:
image.shape

In [ ]:
plt.imshow(image)
plt.title("orig")
plt.show()
for name, converter in converters.items():
    newrgb = np.clip(converter(image[:,:,:3].astype(int)/256),0,1)
    plt.imshow(newrgb)
    plt.title(name)
    plt.show()

In [ ]:
np.sum(image[:,:,:3]<255)/(480*640*4)

In [ ]:
image[np.where(image[:,:,:3]<255)]

In [ ]:
import matplotlib.colors as col

for c in col.to_rgba("skyblue"):
    print(f"{int(256*c):x}", end="\t")

In [ ]:
import matplotlib as mpl
mpl.rc_params()

In [ ]:
import matplotlib.pyplot as plt

print(plt.style.available)


In [ ]:
sns.set_palette("colorblind")
mpl.rc_params("axes")

In [ ]:
x = np.arange(10)
y = np.outer(np.sin(np.arange(10)/2/np.pi),(np.arange(10)/2))
plt.plot(x,y)

In [ ]:
pl = sns.color_palette("colorblind", 2)

In [ ]:
print(pl[0])

In [ ]:
sns.color_palette("colorblind")[0]